In [1]:
# CELL 1: Setup PATH and Imports
import os
import subprocess
import numpy as np
import nibabel as nib
from pathlib import Path
import psutil
import scipy.io as sio
from scipy.stats import spearmanr, pearsonr,kendalltau
from scipy.spatial.distance import pdist, squareform
from sklearn.metrics.pairwise import cosine_similarity 
import pandas as pd
# Add Connectome Workbench to PATH
os.environ['PATH'] = '/opt/workbench/bin_linux64:' + os.environ['PATH']

## Searchlight RSA

In [2]:
# ============================================================================
# TIMING AND SEGMENTATION FUNCTIONS
# ============================================================================

def average_2s_bins_delay(X, TR, onset_sec, duration_sec, bin_sec=2.0, delay_sec=4.0):
    """
    Extract and bin fMRI data with BOLD delay compensation.
    
    Parameters:
    -----------
    X : np.ndarray
        fMRI data (n_vertices, n_timepoints)
    TR : float
        Repetition time in seconds
    onset_sec : float
        Stimulus onset in seconds
    duration_sec : float
        Stimulus duration in seconds
    bin_sec : float
        Bin size in seconds (default 2.0)
    delay_sec : float
        BOLD delay in seconds (default 4.0)

    Returns:
    --------
    Y : np.ndarray
        Binned data (n_vertices, n_bins)
    """
    n_vertices, n_timepoints = X.shape

    # Apply delay to the stimulus window
    onset_d = onset_sec + delay_sec
    end_d = onset_sec + duration_sec + delay_sec

    # Convert to timepoint indices
    start_tp = int(np.floor(onset_d / TR))
    end_tp = int(np.ceil(end_d / TR))

    # Clip to valid range
    start_tp = max(start_tp, 0)
    end_tp = min(end_tp, n_timepoints)

    if end_tp <= start_tp:
        return np.zeros((n_vertices, 0), dtype=X.dtype)

    segment = X[:, start_tp:end_tp]

    # Timepoints per bin
    tp_per_bin = int(np.round(bin_sec / TR))
    if tp_per_bin < 1:
        raise ValueError(f"bin_sec/TR is < 1 (bin_sec={bin_sec}, TR={TR})")

    # Drop leftover timepoints that don't fill a full bin
    n_bins = segment.shape[1] // tp_per_bin
    segment = segment[:, :n_bins * tp_per_bin]

    if n_bins == 0:
        return np.zeros((n_vertices, 0), dtype=X.dtype)

    Y = segment.reshape(n_vertices, n_bins, tp_per_bin).mean(axis=2)
    return Y


def process_all_video_segments(fmri_data, timing_df, TR=1.0, bin_sec=2.0, delay_sec=4.0):
    """
    Process all video segments according to timing file.
    
    Parameters:
    -----------
    fmri_data : np.ndarray
        Full fMRI data (n_vertices, n_timepoints)
    timing_df : pd.DataFrame
        DataFrame with columns: video_id, onset_sec, duration_sec
    TR : float
        Repetition time in seconds
    bin_sec : float
        Bin size in seconds
    delay_sec : float
        BOLD delay in seconds
    
    Returns:
    --------
    processed_fmri : np.ndarray
        Concatenated binned data (n_vertices, total_bins)
    """
    import pandas as pd
    
    segments = []
    
    for _, row in timing_df.iterrows():
        binned = average_2s_bins_delay(
            fmri_data, 
            TR=TR,
            onset_sec=row["onset_sec"],
            duration_sec=row["duration_sec"],
            bin_sec=bin_sec,
            delay_sec=delay_sec
        )
        segments.append(binned)
        print(f"  Video {row['video_id']}: {binned.shape[1]} bins")
    
    processed_fmri = np.concatenate(segments, axis=1)
    return processed_fmri

In [3]:
def compute_geodesic_distance_matrix(surface_file, output_dconn, 
                                     limit_mm=None, roi_metric=None,
                                     corrected_areas=None, naive=False):
    """
    Compute full geodesic distance matrix using wb_command's all-to-all function.
    
    Parameters:
    -----------
    surface_file : str
        Path to surface .surf.gii file
    output_dconn : str
        Output path for dconn file (cifti format)
    limit_mm : float, optional
        Stop at specified distance (vertices beyond get -1)
    roi_metric : str, optional
        ROI metric file - only output distances for ROI vertices
    corrected_areas : str, optional
        Vertex areas for group average surface correction
    naive : bool
        Use only neighbors (not recommended, less accurate)
    
    Returns:
    --------
    output_dconn : str
        Path to output dconn file
    """
    print(f"Computing all-to-all geodesic distances...")
    print(f"Surface: {surface_file}")
    print(f"Output: {output_dconn}")
    
    # Build command
    cmd = [
        'wb_command',
        '-surface-geodesic-distance-all-to-all',
        surface_file,
        output_dconn
    ]
    
    # Add optional parameters
    if roi_metric is not None:
        cmd.extend(['-roi', roi_metric])
        print(f"Using ROI: {roi_metric}")
    
    if limit_mm is not None:
        cmd.extend(['-limit', str(limit_mm)])
        print(f"Distance limit: {limit_mm} mm")
    
    if corrected_areas is not None:
        cmd.extend(['-corrected-areas', corrected_areas])
        print(f"Using corrected areas: {corrected_areas}")
    
    if naive:
        cmd.append('-naive')
        print("Using naive mode (neighbors only)")
    
    # Run command
    print("Running wb_command...")
    result = subprocess.run(cmd, capture_output=True, text=True)
    
    if result.returncode != 0:
        print(f"ERROR: {result.stderr}")
        raise RuntimeError(f"wb_command failed: {result.stderr}")
    
    print(f"Success! Distance matrix saved to {output_dconn}")
    return output_dconn


def load_dconn_as_matrix(dconn_file):
    """
    Load dconn file and convert to numpy matrix.
    
    Parameters:
    -----------
    dconn_file : str
        Path to dconn file from geodesic distance computation
    
    Returns:
    --------
    distance_matrix : np.ndarray
        NxN distance matrix
    """
    print(f"Loading distance matrix from {dconn_file}...")
    
    # Load cifti file
    cifti = nib.load(dconn_file)
    
    # Extract data matrix
    distance_matrix = cifti.get_fdata()
    
    print(f"Distance matrix shape: {distance_matrix.shape}")
    print(f"Distance matrix size in memory: {distance_matrix.nbytes / (1024**3):.2f} GB")
    
    return distance_matrix


def get_k_nearest_neighbors(distance_matrix, k=100):
    """
    Get K nearest neighbors for each vertex from distance matrix.
    
    Parameters:
    -----------
    distance_matrix : np.ndarray or str
        Either NxN distance matrix or path to .npy/.dconn file
    k : int
        Number of nearest neighbors (including center vertex)
    
    Returns:
    --------
    neighbor_indices : np.ndarray
        Shape (N, k) - indices of k nearest neighbors for each vertex
    """
    # Load if path provided
    if isinstance(distance_matrix, (str, Path)):
        if str(distance_matrix).endswith('.npy'):
            distance_matrix = np.load(distance_matrix)
        else:  # assume dconn
            distance_matrix = load_dconn_as_matrix(distance_matrix)
    
    n_vertices = distance_matrix.shape[0]
    print(f"Finding {k} nearest neighbors for {n_vertices} vertices...")
    
    # Get k nearest for each vertex
    neighbor_indices = np.zeros((n_vertices, k), dtype=np.int32)
    
    for i in range(n_vertices):
        if i % 5000 == 0:
            print(f"Progress: {i}/{n_vertices}")
        
        # Get sorted indices by distance
        sorted_idx = np.argsort(distance_matrix[i, :])
        neighbor_indices[i, :] = sorted_idx[:k]
    
    print("Done!")
    return neighbor_indices

In [4]:
def load_model_rdms(model_file):
    """
    Load pre-computed model RDMs or embeddings. if embeddings, compute RDM. 
    
    Parameters:
    -----------
    model_file : str
        Path to model data (.npy/.npz). 
        Can be either:
        - RDMs: shape (n_timepoints, n_timepoints) - pre-computed dissimilarity matrix
        - Embeddings: shape (n_timepoints, n_features) - will compute RDM from this
    
    Returns:
    --------
    model_rdm : np.ndarray
        RDM for the model (n_timepoints, n_timepoints)
    """
    
    print(f"Loading model data from {model_file}...")
    
    model_data = np.load(model_file)
    
    if model_data.ndim == 2:
        # Check if it's already an RDM (square matrix)
        if model_data.shape[0] == model_data.shape[1]:
            print(f"  Loaded pre-computed RDM: {model_data.shape}")
            return model_data
        else:
            # It's embeddings, compute RDM using correlation distance
            print(f"  Computing RDM from embeddings: {model_data.shape}")
            rdm  = squareform(pdist(model_data, metric='correlation'))
            print(f"  RDM computed with shape: {rdm.shape}")
            return rdm
    else:
        raise ValueError(f"Unexpected model data shape: {model_data.shape}")


def save_as_gifti_metric(data, output_file, metric_name="metric"):
    """
    Save numpy array as GIFTI metric file for Workbench visualization.
    
    Parameters:
    -----------
    data : np.ndarray
        1D array (n_vertices,)
    output_file : str
        Output .func.gii path
    metric_name : str
        Name for the metric
    """
    # Create single darray
    darray = nib.gifti.GiftiDataArray(
        data=data.astype(np.float32),
        intent='NIFTI_INTENT_NONE',
        datatype='NIFTI_TYPE_FLOAT32'
    )
    
    # Add metadata
    meta = nib.gifti.GiftiMetaData()
    meta['Name'] = metric_name
    darray.meta = meta
    
    # Create and save
    gii = nib.GiftiImage(darrays=[darray])
    nib.save(gii, output_file)
    print(f"Saved GIFTI metric: {output_file}")





def compute_searchlight_rsa_map(fmri_data, neighbor_indices, model_rdm,
                                output_file=None, method='spearman',
                                save_pvalues=True,
                                min_variance=1e-10):
    """
    Compute RSA correlation map for all searchlights.
    
    CORRECTED VERSION with proper distance metrics following Kriegeskorte et al.
    
    Parameters:
    -----------
    fmri_data : np.ndarray
        fMRI data (n_vertices, n_timepoints)
    neighbor_indices : np.ndarray
        Shape (n_vertices, k) - neighbor indices for each searchlight
    model_rdm : np.ndarray or str
        Either pre-computed RDM or path to model file
    output_file : str, optional
        Path to save correlation map
    method : str
        'spearman' (recommended), 'pearson', or 'kendall' for RSA comparison
    save_pvalues : bool
        Whether to also save p-values
    min_variance : float
        Minimum variance threshold to skip constant searchlights
    
    Returns:
    --------
    rsa_correlations : np.ndarray
        Correlation values (n_vertices,) - NaN for skipped vertices
    rsa_pvalues : np.ndarray (optional)
        P-values (n_vertices,) - NaN for skipped vertices
    
    Notes:
    ------
    Following Kriegeskorte et al. (2008, 2013) recommendations:
    - Brain RDM: correlation distance (standard for fMRI)
    - RSA comparison: Spearman rank correlation (handles non-linearity)
    """
    
    n_vertices, n_timepoints = fmri_data.shape
    print(f"fMRI data shape: {n_vertices} vertices × {n_timepoints} timepoints")
    
    # Load model RDM if path provided
    if isinstance(model_rdm, (str, Path)):
        print(f"Loading model data from {model_rdm}...")
        model_data = np.load(model_rdm)
        
        if model_data.ndim == 2:
            if model_data.shape[0] == model_data.shape[1]:
                print(f"  Loaded pre-computed RDM: {model_data.shape}")
                model_rdm = model_data
            else:
                print(f"  Computing RDM from embeddings: {model_data.shape}")
                # Use correlation distance for consistency with brain RDM
                print(f"  Using correlation distance for model RDM (Kriegeskorte recommendation)")
                model_rdm = squareform(pdist(model_data, metric='correlation'))
                print(f"  RDM computed with shape: {model_rdm.shape}")
    
    print(f"Model RDM shape: {model_rdm.shape}")
    
    # Verify model RDM matches timepoints
    if model_rdm.shape[0] != n_timepoints:
        raise ValueError(
            f"Model RDM timepoints ({model_rdm.shape[0]}) doesn't match "
            f"fMRI timepoints ({n_timepoints})"
        )
    
    # Vectorize model RDM once (upper triangle)
    triu_indices = np.triu_indices(n_timepoints, k=1)
    model_rdm_vec = model_rdm[triu_indices]
    
    print(f"\nModel RDM statistics:")
    print(f"  Vectorized: {len(model_rdm_vec)} unique pairs")
    print(f"  Range: [{model_rdm_vec.min():.4f}, {model_rdm_vec.max():.4f}]")
    print(f"  Mean: {model_rdm_vec.mean():.4f}")
    print(f"  Std: {model_rdm_vec.std():.4f}")
    
    # Check if model RDM has variance
    if model_rdm_vec.std() < min_variance:
        raise ValueError("Model RDM has zero variance! Cannot compute correlations.")
    
    # Initialize output arrays with NaN (for skipped vertices)
    n_searchlights = neighbor_indices.shape[0]
    rsa_correlations = np.full(n_searchlights, np.nan, dtype=np.float32)
    
    if save_pvalues:
        rsa_pvalues = np.full(n_searchlights, np.nan, dtype=np.float32)
    
    print(f"\nComputing RSA for {n_searchlights} searchlights...")
    print(f"RSA comparison method: {method}")
    
    # Track statistics
    n_computed = 0
    n_skipped = 0
    n_failed = 0
    
    # Main loop - compute RSA for each searchlight
    for vertex_idx in range(n_searchlights):
        if vertex_idx % 1000 == 0:
            print(f"Progress: {vertex_idx}/{n_searchlights} ({100*vertex_idx/n_searchlights:.1f}%) | "
                  f"Computed: {n_computed}, Skipped: {n_skipped}, Failed: {n_failed}")
        
        try:
            # Get neighbor activations for this searchlight
            neighbors = neighbor_indices[vertex_idx, :]
            searchlight_activations = fmri_data[neighbors, :].T  # (n_timepoints, k_neighbors)
            
            # Check for sufficient variance
            if searchlight_activations.std() < min_variance:
                n_skipped += 1
                continue
            
            # Compute brain RDM using correlation:  Invariant to linear scaling and constant offsets
            searchlight_rdm = squareform(pdist(searchlight_activations, metric='correlation'))
            # Vectorize searchlight RDM
            searchlight_rdm_vec = searchlight_rdm[triu_indices]
            
            # Check if searchlight RDM has variance
            if searchlight_rdm_vec.std() < min_variance:
                n_skipped += 1
                continue
            
            # Compute correlation with model RDM
            # RECOMMENDED: Spearman rank correlation (handles non-linearity)
            if method == 'spearman':
                corr, pval = spearmanr(searchlight_rdm_vec, model_rdm_vec, nan_policy='omit')
            elif method == 'pearson':
                corr, pval = pearsonr(searchlight_rdm_vec, model_rdm_vec)
            elif method == 'kendall':
                corr, pval = kendalltau(searchlight_rdm_vec, model_rdm_vec, nan_policy='omit')
            else:
                raise ValueError(f"Unknown method: {method}")
            
            # Check if correlation is valid
            if np.isnan(corr) or np.isnan(pval):
                n_failed += 1
                continue
            
            # Store results
            rsa_correlations[vertex_idx] = corr
            if save_pvalues:
                rsa_pvalues[vertex_idx] = pval
            
            n_computed += 1
            
        except Exception as e:
            n_failed += 1
            if n_failed <= 5:  # Only print first few errors
                print(f"  Error at vertex {vertex_idx}: {str(e)}")
            continue
    
    print(f"\nRSA computation complete!")
    print(f"  Successfully computed: {n_computed} / {n_searchlights} ({100*n_computed/n_searchlights:.1f}%)")
    print(f"  Skipped (low variance): {n_skipped} ({100*n_skipped/n_searchlights:.1f}%)")
    print(f"  Failed (errors): {n_failed} ({100*n_failed/n_searchlights:.1f}%)")
    
    # Report statistics on valid correlations
    valid_corr = rsa_correlations[~np.isnan(rsa_correlations)]
    if len(valid_corr) > 0:
        print(f"\nValid correlation statistics:")
        print(f"  Range: [{valid_corr.min():.4f}, {valid_corr.max():.4f}]")
        print(f"  Mean: {valid_corr.mean():.4f} ± {valid_corr.std():.4f}")
        print(f"  Median: {np.median(valid_corr):.4f}")
        
        if save_pvalues:
            valid_pval = rsa_pvalues[~np.isnan(rsa_pvalues)]
            n_sig_05 = (valid_pval < 0.05).sum()
            n_sig_01 = (valid_pval < 0.01).sum()
            n_sig_001 = (valid_pval < 0.001).sum()
            print(f"\nSignificance (uncorrected p-values):")
            print(f"  p < 0.05:  {n_sig_05} / {len(valid_pval)} ({100*n_sig_05/len(valid_pval):.1f}%)")
            print(f"  p < 0.01:  {n_sig_01} / {len(valid_pval)} ({100*n_sig_01/len(valid_pval):.1f}%)")
            print(f"  p < 0.001: {n_sig_001} / {len(valid_pval)} ({100*n_sig_001/len(valid_pval):.1f}%)")
            print(f"\n  ⚠ Note: These are uncorrected p-values.")
            print(f"    Consider FDR correction for {n_searchlights} multiple comparisons.")
    else:
        print("\n⚠️ WARNING: No valid correlations computed!")
        print("   Check your data for issues (constant values, NaNs, etc.)")
    
    # Save results
    if output_file:
        output_path = Path(output_file)
        
        # Save correlations
        np.save(output_path.with_suffix('.npy'), rsa_correlations)
        print(f"\nSaved correlations to {output_path.with_suffix('.npy')}")
        
        if save_pvalues:
            pval_file = output_path.with_name(output_path.stem + '_pvalues.npy')
            np.save(pval_file, rsa_pvalues)
            print(f"Saved p-values to {pval_file}")
        
        # Save as GIFTI metric for visualization (replace NaN with 0)
        corr_for_viz = np.nan_to_num(rsa_correlations, nan=0.0)
        
        darray = nib.gifti.GiftiDataArray(
            data=corr_for_viz.astype(np.float32),
            intent='NIFTI_INTENT_NONE',
            datatype='NIFTI_TYPE_FLOAT32'
        )
        
        meta = nib.gifti.GiftiMetaData()
        meta['Name'] = f"RSA_{method}"
        darray.meta = meta
        
        gii = nib.GiftiImage(darrays=[darray])
        nib.save(gii, str(output_path.with_suffix('.func.gii')))
        print(f"Saved GIFTI metric to {output_path.with_suffix('.func.gii')}")
        print(f"  (NaN values set to 0 for visualization)")
    
    if save_pvalues:
        return rsa_correlations, rsa_pvalues
    else:
        return rsa_correlations



In [5]:
# ============================================================================
# COMPLETE PIPELINE FUNCTION
# ============================================================================

def complete_rsa_pipeline(surface_file, fmri_data, model_file, output_dir,
                          n_neighbors=100, limit_mm=None, method='spearman',
                          ):
    """
    Complete memory-efficient RSA pipeline.
    
    Parameters:
    -----------
    surface_file : str
        Surface geometry file
    fmri_data : np.ndarray
        Preprocessed fMRI data (n_vertices, n_timepoints)
    model_file : str
        Model RDM or embeddings (.npy file)
    output_dir : str
        Output directory
    n_neighbors : int
        Searchlight size
    limit_mm : float, optional
        Distance limit in mm
    method : str
        'spearman' or 'pearson' or 'kendall'
    """
    output_dir = Path(output_dir)
    output_dir.mkdir(parents=True, exist_ok=True)
    
    print("=" * 70)
    print("SEARCHLIGHT RSA PIPELINE")
    print("=" * 70)
    
    # Step 1: Compute geodesic distances
    print("\nSTEP 1: Computing geodesic distances...")
    print("-" * 70)
    
    dconn_file = output_dir / "geodesic_distances.dconn.nii"
    
    if not dconn_file.exists():
        compute_geodesic_distance_matrix(
            surface_file=surface_file,
            output_dconn=str(dconn_file),
            limit_mm=limit_mm
        )
    else:
        print(f"Using existing distance file: {dconn_file}")
    
    # Step 2: Get neighbors
    print(f"\nSTEP 2: Finding {n_neighbors} nearest neighbors...")
    print("-" * 70)
    
    neighbors_file = output_dir / f"neighbor_indices_k{n_neighbors}.npy"
    
    if not neighbors_file.exists():
        distance_matrix = load_dconn_as_matrix(str(dconn_file))
        neighbor_indices = get_k_nearest_neighbors(distance_matrix, k=n_neighbors)
        np.save(neighbors_file, neighbor_indices)
        print(f"Saved neighbor indices to {neighbors_file}")
    else:
        print(f"Loading existing neighbor indices: {neighbors_file}")
        neighbor_indices = np.load(neighbors_file)
    
    # Step 3: Compute RSA map
    print("\nSTEP 3: Computing RSA correlation map...")
    print("-" * 70)
    
    rsa_output = output_dir / f"rsa_map_{method}_k{n_neighbors}"
    
    rsa_correlations, rsa_pvalues = compute_searchlight_rsa_map(
        fmri_data=fmri_data,
        neighbor_indices=neighbor_indices,
        model_rdm=model_file,
        output_file=str(rsa_output),
        method=method,
        save_pvalues=True,
        min_variance=1e-10
    )
    
    print("\n" + "=" * 70)
    print("PIPELINE COMPLETE!")
    print("=" * 70)
    print(f"Results saved in: {output_dir}")
    print(f"  RSA correlation map: {rsa_output}.func.gii")
    print(f"  RSA p-values: {rsa_output}_pvalues.npy")
    print(f"\nVisualize in Workbench:")
    print(f"  wb_view {surface_file} {rsa_output}.func.gii")
    
    return rsa_correlations, rsa_pvalues

In [6]:
# Configuration
DATA_DIR = Path("/home/amin/Research/Representation/Movie/data/Setareh")
HCP_DATA_DIR = DATA_DIR / "HCP Data"
SURFACE_DIR = DATA_DIR / "HCP_S1200_GroupAvg_v1"
OUTPUT_DIR = Path("/home/amin/Research/Representation/Movie/outputs/searchlight_rsa_output")

# Input files
mat_file_left = HCP_DATA_DIR / "notmean_left_Meanfile.mat"
mat_file_right = HCP_DATA_DIR / "notmean_right_Meanfile.mat"

surface_left = SURFACE_DIR / "S1200.L.midthickness_MSMAll.32k_fs_LR.surf.gii"
surface_right = SURFACE_DIR / "S1200.R.midthickness_MSMAll.32k_fs_LR.surf.gii"

# ADD TIMING FILE
timing_csv = DATA_DIR / "Data/movie_timing.csv"  

# Model files
model_file = Path("/home/amin/Research/Representation/Movie/outputs/model_embeddings/pe-av-small/peav_small_16frame_av.npy")

# Parameters
N_NEIGHBORS = 100
LIMIT_MM = None
RSA_METHOD = 'spearman'
TR = 1.0  
BIN_SEC = 2.0 
DELAY_SEC = 4.0 

# ============================================================================
# STEP 0: Process timing and segment fMRI data
# ============================================================================
print("=" * 70)
print("STEP 0: Processing fMRI with timing/segmentation")
print("=" * 70)

timing_df = pd.read_csv(timing_csv)
print(f"\nTiming information loaded:")
print(timing_df)

# Process LEFT hemisphere
print("\n--- Processing LEFT hemisphere ---")
mat_data_left = sio.loadmat(mat_file_left)
fmri_left_raw = mat_data_left['Left_data']
if fmri_left_raw.shape[0] != 32492:
    fmri_left_raw = fmri_left_raw.T

print(f"Raw fMRI shape: {fmri_left_raw.shape}")
fmri_left_processed = process_all_video_segments(
    fmri_left_raw, timing_df, TR=TR, bin_sec=BIN_SEC, delay_sec=DELAY_SEC
)
print(f"Processed fMRI shape: {fmri_left_processed.shape}")

# Process RIGHT hemisphere
print("\n--- Processing RIGHT hemisphere ---")
mat_data_right = sio.loadmat(mat_file_right)
fmri_right_raw = mat_data_right['Right_data']
if fmri_right_raw.shape[0] != 32492:
    fmri_right_raw = fmri_right_raw.T

print(f"Raw fMRI shape: {fmri_right_raw.shape}")
fmri_right_processed = process_all_video_segments(
    fmri_right_raw, timing_df, TR=TR, bin_sec=BIN_SEC, delay_sec=DELAY_SEC
)
print(f"Processed fMRI shape: {fmri_right_processed.shape}")


# ============================================================================
# STEP 1-3: Run RSA pipeline for LEFT hemisphere
# ============================================================================
print("\n" + "=" * 70)
print("PROCESSING LEFT HEMISPHERE")
print("=" * 70)

rsa_corr_left, rsa_pval_left = complete_rsa_pipeline(
    surface_file=str(surface_left),
    fmri_data=fmri_left_processed,
    model_file=str(model_file),
    output_dir=str(OUTPUT_DIR / "left_hemisphere"),
    n_neighbors=N_NEIGHBORS,
    limit_mm=LIMIT_MM,
    method=RSA_METHOD
)

# ============================================================================
# STEP 1-3: Run RSA pipeline for RIGHT hemisphere
# ============================================================================
print("\n" + "=" * 70)
print("PROCESSING RIGHT HEMISPHERE")
print("=" * 70)

rsa_corr_right, rsa_pval_right = complete_rsa_pipeline(
    surface_file=str(surface_right),
    fmri_data=fmri_right_processed,
    model_file=str(model_file),
    output_dir=str(OUTPUT_DIR / "right_hemisphere"),
    n_neighbors=N_NEIGHBORS,
    limit_mm=LIMIT_MM,
    method=RSA_METHOD
)

# ============================================================================
# SUMMARY & STATISTICS
# ============================================================================
print("\n" + "=" * 70)
print("ALL PROCESSING COMPLETE!")
print("=" * 70)

print(f"\nOutput directory: {OUTPUT_DIR}")

print(f"\n{'='*70}")
print("LEFT HEMISPHERE RESULTS:")
print(f"{'='*70}")
print(f"  RSA correlation range: [{rsa_corr_left.min():.4f}, {rsa_corr_left.max():.4f}]")
print(f"  Mean correlation: {rsa_corr_left.mean():.4f} ± {rsa_corr_left.std():.4f}")
print(f"  Median correlation: {np.median(rsa_corr_left):.4f}")
print(f"  Significant vertices (p < 0.05): {(rsa_pval_left < 0.05).sum():,} / {len(rsa_pval_left):,} ({100*(rsa_pval_left < 0.05).sum()/len(rsa_pval_left):.1f}%)")
print(f"  Significant vertices (p < 0.01): {(rsa_pval_left < 0.01).sum():,} / {len(rsa_pval_left):,} ({100*(rsa_pval_left < 0.01).sum()/len(rsa_pval_left):.1f}%)")
print(f"  Significant vertices (p < 0.001): {(rsa_pval_left < 0.001).sum():,} / {len(rsa_pval_left):,} ({100*(rsa_pval_left < 0.001).sum()/len(rsa_pval_left):.1f}%)")

print(f"\n{'='*70}")
print("RIGHT HEMISPHERE RESULTS:")
print(f"{'='*70}")
print(f"  RSA correlation range: [{rsa_corr_right.min():.4f}, {rsa_corr_right.max():.4f}]")
print(f"  Mean correlation: {rsa_corr_right.mean():.4f} ± {rsa_corr_right.std():.4f}")
print(f"  Median correlation: {np.median(rsa_corr_right):.4f}")
print(f"  Significant vertices (p < 0.05): {(rsa_pval_right < 0.05).sum():,} / {len(rsa_pval_right):,} ({100*(rsa_pval_right < 0.05).sum()/len(rsa_pval_right):.1f}%)")
print(f"  Significant vertices (p < 0.01): {(rsa_pval_right < 0.01).sum():,} / {len(rsa_pval_right):,} ({100*(rsa_pval_right < 0.01).sum()/len(rsa_pval_right):.1f}%)")
print(f"  Significant vertices (p < 0.001): {(rsa_pval_right < 0.001).sum():,} / {len(rsa_pval_right):,} ({100*(rsa_pval_right < 0.001).sum()/len(rsa_pval_right):.1f}%)")

# ============================================================================
# VISUALIZATION COMMANDS
# ============================================================================
print(f"\n{'='*70}")
print("VISUALIZATION IN CONNECTOME WORKBENCH:")
print(f"{'='*70}")
print("\nLeft hemisphere:")
print(f"  wb_view {surface_left} \\")
print(f"    {OUTPUT_DIR}/left_hemisphere/rsa_map_{RSA_METHOD}_k{N_NEIGHBORS}.func.gii")

print("\nRight hemisphere:")
print(f"  wb_view {surface_right} \\")
print(f"    {OUTPUT_DIR}/right_hemisphere/rsa_map_{RSA_METHOD}_k{N_NEIGHBORS}.func.gii")

print("\nBoth hemispheres together:")
print(f"  wb_view {surface_left} \\")
print(f"    {OUTPUT_DIR}/left_hemisphere/rsa_map_{RSA_METHOD}_k{N_NEIGHBORS}.func.gii \\")
print(f"    {surface_right} \\")
print(f"    {OUTPUT_DIR}/right_hemisphere/rsa_map_{RSA_METHOD}_k{N_NEIGHBORS}.func.gii")

# ============================================================================
# SAVE COMBINED SUMMARY
# ============================================================================
summary_file = OUTPUT_DIR / "rsa_analysis_summary.txt"
with open(summary_file, 'w') as f:
    f.write("=" * 70 + "\n")
    f.write("SEARCHLIGHT RSA ANALYSIS SUMMARY\n")
    f.write("=" * 70 + "\n\n")
    f.write(f"Searchlight size: {N_NEIGHBORS} neighbors\n")
    f.write(f"RSA method: {RSA_METHOD}\n")
    f.write(f"Distance limit: {LIMIT_MM if LIMIT_MM else 'None (unlimited)'}\n\n")
    
    f.write("LEFT HEMISPHERE:\n")
    f.write(f"  Mean correlation: {rsa_corr_left.mean():.4f}\n")
    f.write(f"  Significant (p<0.05): {(rsa_pval_left < 0.05).sum()} vertices\n\n")
    
    f.write("RIGHT HEMISPHERE:\n")
    f.write(f"  Mean correlation: {rsa_corr_right.mean():.4f}\n")
    f.write(f"  Significant (p<0.05): {(rsa_pval_right < 0.05).sum()} vertices\n")

print(f"\nSummary saved to: {summary_file}")

print("\n" + "=" * 70)
print("ANALYSIS COMPLETE! 🎉")
print("=" * 70)

STEP 0: Processing fMRI with timing/segmentation

Timing information loaded:
   video_id  onset_sec  end_sec  duration_sec
0    video1         20      232           212
1    video2        284      504           220
2    video3        526      712           186
3    video4        735      797            62
4    video5        818      900            82
5    video6        920     1146           226
6    video7       1167     1425           258
7    video8       1446     1694           248
8    video9       1716     1798            82
9   video10       1818     1998           180
10  video11       2019     2203           184
11  video12       2224     2426           202
12  video13       2449     2589           140
13  video14       2611     2693            82
14  video15       2713     2945           232
15  video16       2965     3195           230
16  video17       3215     3469           254
17  video18       3492     3574            82

--- Processing LEFT hemisphere ---
Raw fMRI shap